# AutoInt+ 실험

In [5]:
# 1. 라이브러리 임포트
import time
import random
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, MaxPooling2D, Conv2D, Dropout, Lambda, Dense, Flatten, Activation, Input, Embedding, BatchNormalization
from tensorflow.keras.initializers import glorot_normal, Zeros, TruncatedNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import BinaryAccuracy
from collections import defaultdict
import joblib
import math

In [10]:
# 2. 레이어 정의 (Embedding, MLP, Attention, AutoIntMLP)
class FeaturesEmbedding(Layer):
    '''
    임베딩 레이어입니다. 
    '''
    def __init__(self, field_dims, embed_dim, **kwargs):
        super(FeaturesEmbedding, self).__init__(**kwargs)
        self.total_dim = sum(field_dims)
        self.embed_dim = embed_dim
        # 호환성을 위해 int32로 유지
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.int32)
        self.embedding = tf.keras.layers.Embedding(input_dim=self.total_dim, output_dim=self.embed_dim)

    def build(self, input_shape):
        self.embedding.build(input_shape)
        self.embedding.set_weights([tf.keras.initializers.GlorotUniform()(shape=self.embedding.weights[0].shape)])

    def call(self, x):
        # [수정 핵심] 입력값(x)가 int64로 들어오면 int32인 offsets와 계산이 안되므로 형변환 수행
        x = tf.cast(x, dtype=tf.int32)
        x = x + tf.constant(self.offsets)
        return self.embedding(x)

class MultiLayerPerceptron(Layer):
    '''
    DNN(MLP) 레이어
    '''
    def __init__(self, input_dim, hidden_units, activation='relu', l2_reg=0, dropout_rate=0, use_bn=False, init_std=0.0001, output_layer=True):
        super(MultiLayerPerceptron, self).__init__()
        self.dropout_rate = dropout_rate
        self.use_bn = use_bn
        hidden_units = [input_dim] + list(hidden_units)
        if output_layer:
            hidden_units += [1]
        
        self.linears = [Dense(units, activation=None, kernel_initializer=tf.random_normal_initializer(stddev=init_std),
                              kernel_regularizer=tf.keras.regularizers.l2(l2_reg)) for units in hidden_units[1:]]
        self.activation = tf.keras.layers.Activation(activation)
        if self.use_bn:
            self.bn = [BatchNormalization() for _ in hidden_units[1:]]
        self.dropout = Dropout(dropout_rate)

    def call(self, inputs, training=False):
        x = inputs
        for i in range(len(self.linears)):
            x = self.linears[i](x)
            if self.use_bn:
                x = self.bn[i](x, training=training)
            x = self.activation(x)
            x = self.dropout(x, training=training)
        return x

class MultiHeadSelfAttention(Layer):
    '''
    멀티 헤드 셀프 어텐션 레이어
    '''
    def __init__(self, att_embedding_size=8, head_num=2, use_res=True, scaling=False, seed=1024, **kwargs):
        if head_num <= 0:
            raise ValueError('head_num must be a int > 0')
        self.att_embedding_size = att_embedding_size
        self.head_num = head_num
        self.use_res = use_res
        self.seed = seed
        self.scaling = scaling
        super(MultiHeadSelfAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        if len(input_shape) != 3:
            raise ValueError("Unexpected inputs dimensions %d, expect to be 3 dimensions" % (len(input_shape)))
        embedding_size = int(input_shape[-1])
        self.W_Query = self.add_weight(name='query', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                       dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed))
        self.W_key = self.add_weight(name='key', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                     dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed + 1))
        self.W_Value = self.add_weight(name='value', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                       dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed + 2))
        if self.use_res:
            self.W_Res = self.add_weight(name='res', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                         dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed))
        super(MultiHeadSelfAttention, self).build(input_shape)

    def call(self, inputs, **kwargs):
        if K.ndim(inputs) != 3:
            raise ValueError("Unexpected inputs dimensions %d, expect to be 3 dimensions" % (K.ndim(inputs)))
        
        querys = tf.tensordot(inputs, self.W_Query, axes=(-1, 0))
        keys = tf.tensordot(inputs, self.W_key, axes=(-1, 0))
        values = tf.tensordot(inputs, self.W_Value, axes=(-1, 0))

        querys = tf.stack(tf.split(querys, self.head_num, axis=2))
        keys = tf.stack(tf.split(keys, self.head_num, axis=2))
        values = tf.stack(tf.split(values, self.head_num, axis=2))

        inner_product = tf.matmul(querys, keys, transpose_b=True)
        if self.scaling:
            inner_product /= self.att_embedding_size ** 0.5
        self.normalized_att_scores = tf.nn.softmax(inner_product)

        result = tf.matmul(self.normalized_att_scores, values)
        result = tf.concat(tf.split(result, self.head_num, ), axis=-1)
        result = tf.squeeze(result, axis=0)

        if self.use_res:
            result += tf.tensordot(inputs, self.W_Res, axes=(-1, 0))
        result = tf.nn.relu(result)
        return result

    def compute_output_shape(self, input_shape):
        return (None, input_shape[1], self.att_embedding_size * self.head_num)

    def get_config(self, ):
        config = {'att_embedding_size': self.att_embedding_size, 'head_num': self.head_num, 'use_res': self.use_res,'seed': self.seed}
        base_config = super(MultiHeadSelfAttention, self).get_config()
        base_config.update(config)
        return base_config

class AutoIntMLP(Layer):
    '''
    AutoInt + MLP 결합 레이어
    '''
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False, dnn_dropout=0.4, init_std=0.0001):
        super(AutoIntMLP, self).__init__()
        self.embedding = FeaturesEmbedding(field_dims, embedding_size)
        self.num_fields = len(field_dims)
        self.embedding_size = embedding_size

        self.final_layer = Dense(1, use_bias=False, kernel_initializer=tf.random_normal_initializer(stddev=init_std))
        
        # DNN 부분 구성
        self.dnn = tf.keras.Sequential()
        for units in dnn_hidden_units:
            self.dnn.add(Dense(units, activation=dnn_activation,
                               kernel_regularizer=tf.keras.regularizers.l2(l2_reg_dnn),
                               kernel_initializer=tf.random_normal_initializer(stddev=init_std)))
            if dnn_use_bn:
                self.dnn.add(BatchNormalization())
            self.dnn.add(Activation(dnn_activation))
            if dnn_dropout > 0:
                self.dnn.add(Dropout(dnn_dropout))
        self.dnn.add(Dense(1, kernel_initializer=tf.random_normal_initializer(stddev=init_std)))

        # Attention 부분 구성
        self.int_layers = [MultiHeadSelfAttention(att_embedding_size=embedding_size, head_num=att_head_num, use_res=att_res) for _ in range(att_layer_num)]

    def call(self, inputs):
        embed_x = self.embedding(inputs)
        # DNN 입력용 Reshape
        dnn_embed = tf.reshape(embed_x, shape=(-1, self.embedding_size * self.num_fields))

        # Attention 연산
        att_input = embed_x
        for layer in self.int_layers:
            att_input = layer(att_input)

        att_output = Flatten()(att_input)
        att_output = self.final_layer(att_output)
        
        # DNN 연산
        dnn_output = self.dnn(dnn_embed)
        
        # 두 결과 합산 후 Sigmoid
        y_pred = tf.keras.activations.sigmoid(att_output + dnn_output)
        
        return y_pred

class AutoIntMLPModel(Model):
    '''
    Keras Model Wrapper
    '''
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2,
                 att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False,
                 dnn_dropout=0.4, init_std=0.0001):
        super(AutoIntMLPModel, self).__init__()
        self.autoInt_layer = AutoIntMLP(
            field_dims=field_dims,
            embedding_size=embedding_size,
            att_layer_num=att_layer_num,
            att_head_num=att_head_num,
            att_res=att_res,
            dnn_hidden_units=dnn_hidden_units,
            dnn_activation=dnn_activation,
            l2_reg_dnn=l2_reg_dnn,
            l2_reg_embedding=l2_reg_embedding,
            dnn_use_bn=dnn_use_bn,
            dnn_dropout=dnn_dropout,
            init_std=init_std
        )

    def call(self, inputs, training=False):
        return self.autoInt_layer(inputs, training=training)

In [7]:
# 3. 평가 지표 및 테스트 함수 정의
def get_DCG(ranklist, y_true):
    dcg = 0.0
    for i in range(len(ranklist)):
        item = ranklist[i]
        if item in y_true:
            dcg += 1.0 / math.log(i + 2)
    return dcg

def get_IDCG(ranklist, y_true):
    idcg = 0.0
    i = 0
    for item in y_true:
        if item in ranklist:
            idcg += 1.0 / math.log(i + 2)
            i += 1
    return idcg

def get_NDCG(ranklist, y_true):
    ranklist = np.array(ranklist).astype(int)
    y_true = np.array(y_true).astype(int)
    dcg = get_DCG(ranklist, y_true)
    idcg = get_IDCG(y_true, y_true)
    if idcg == 0:
        return 0
    return round((dcg / idcg), 5)

def get_hit_rate(ranklist, y_true):
    c = 0
    for y in y_true:
        if y in ranklist:
            c += 1
    return round(c / len(y_true), 5)

def test_model(model, test_df, batch_size=2048):
    '''
    모델 테스트 함수 (스칼라 경고 수정됨)
    '''
    user_pred_info = defaultdict(list)
    total_rows = len(test_df)
    # 마지막 컬럼(Label)을 제외한 Feature만 추출
    for i in range(0, total_rows, batch_size):
        features = test_df.iloc[i:i + batch_size, :-1].values
        y_pred = model.predict(features, verbose=False)
        for feature, p in zip(features, y_pred):
            u_id = int(feature[0]) # user_id
            i_id = int(feature[1]) # movie_id
            
            # 스칼라 값 안전하게 추출
            score = float(p.item() if hasattr(p, 'item') else p[0])
            user_pred_info[u_id].append((i_id, score))
            
    return user_pred_info

In [11]:
# 3. 평가 지표 및 테스트 함수 정의
def get_DCG(ranklist, y_true):
    dcg = 0.0
    for i in range(len(ranklist)):
        item = ranklist[i]
        if item in y_true:
            dcg += 1.0 / math.log(i + 2)
    return dcg

def get_IDCG(ranklist, y_true):
    idcg = 0.0
    i = 0
    for item in y_true:
        if item in ranklist:
            idcg += 1.0 / math.log(i + 2)
            i += 1
    return idcg

def get_NDCG(ranklist, y_true):
    ranklist = np.array(ranklist).astype(int)
    y_true = np.array(y_true).astype(int)
    dcg = get_DCG(ranklist, y_true)
    idcg = get_IDCG(y_true, y_true)
    if idcg == 0:
        return 0
    return round((dcg / idcg), 5)

def get_hit_rate(ranklist, y_true):
    c = 0
    for y in y_true:
        if y in ranklist:
            c += 1
    return round(c / len(y_true), 5)

def test_model(model, test_df, batch_size=2048):
    '''
    모델 테스트 함수
    '''
    user_pred_info = defaultdict(list)
    total_rows = len(test_df)
    # 마지막 컬럼(Label)을 제외한 Feature만 추출
    for i in range(0, total_rows, batch_size):
        features = test_df.iloc[i:i + batch_size, :-1].values
        y_pred = model.predict(features, verbose=False)
        for feature, p in zip(features, y_pred):
            u_id = int(feature[0]) # user_id
            i_id = int(feature[1]) # movie_id
            
            # 스칼라 값 안전하게 추출
            score = float(p.item() if hasattr(p, 'item') else p[0])
            user_pred_info[u_id].append((i_id, score))
            
    return user_pred_info

In [12]:
# 4. 데이터 로드 및 전처리
# 경로 설정
data_path = '~/aiffel/autoint/ml-1m'
# 데이터 로드
movielens_rcmm = pd.read_csv(f"{data_path}/movielens_rcmm_v2.csv", dtype=str)
print("Data Shape:", movielens_rcmm.shape)

# 라벨 인코딩
label_encoders = {col: LabelEncoder() for col in movielens_rcmm.columns[:-1]} # label 컬럼 제외
for col, le in label_encoders.items():
    movielens_rcmm[col] = le.fit_transform(movielens_rcmm[col])

# 라벨 타입 변환
movielens_rcmm['label'] = movielens_rcmm['label'].astype(np.float32)

# 학습/테스트 분리
train_df, test_df = train_test_split(movielens_rcmm, test_size=0.2, random_state=42)

# Field Dims 계산 (Feature 차원 수)
u_i_feature = ['user_id', 'movie_id']
meta_features = ['movie_decade', 'movie_year', 'rating_year', 'rating_month', 'rating_decade', 'genre1','genre2', 'genre3', 'gender', 'age', 'occupation', 'zip']
label = 'label'
field_dims = np.max(movielens_rcmm[u_i_feature + meta_features].astype(np.int64).values, axis=0) + 1
print("Field Dims:", field_dims)

Data Shape: (1000209, 15)
Field Dims: [6040 3706   10   81    4   12    1   18   18   16    2    7   21 3439]


In [13]:
# 5. 모델 학습 환경 설정 및 훈련
# 하이퍼파라미터
epochs = 5
learning_rate = 0.0001
dropout = 0.4
batch_size = 2048
embed_dim = 16

# AutoIntMLP 모델 생성
autoIntMLP_model = AutoIntMLPModel(
    field_dims=field_dims,
    embedding_size=embed_dim,
    att_layer_num=3,
    att_head_num=2,
    att_res=True,
    dnn_hidden_units=(32, 32),    # DNN 구조
    dnn_activation='relu',
    l2_reg_dnn=0,
    l2_reg_embedding=1e-5,
    dnn_use_bn=False,
    dnn_dropout=dropout,
    init_std=0.0001
)

# 컴파일
optimizer = Adam(learning_rate=learning_rate)
loss_fn = BinaryCrossentropy(from_logits=False)
autoIntMLP_model.compile(optimizer=optimizer, loss=loss_fn, metrics=['binary_crossentropy'])

# 훈련 시작
print("\n========== Training Start ==========")
history = autoIntMLP_model.fit(
    train_df[u_i_feature + meta_features], 
    train_df[label], 
    epochs=epochs, 
    batch_size=batch_size, 
    validation_split=0.1
)


========== Training Start ==========
Epoch 1/5
352/352 [==============================] - 8s 15ms/step - loss: 0.6737 - binary_crossentropy: 0.6737 - val_loss: 0.6320 - val_binary_crossentropy: 0.6320
Epoch 2/5
352/352 [==============================] - 5s 14ms/step - loss: 0.6030 - binary_crossentropy: 0.6030 - val_loss: 0.5882 - val_binary_crossentropy: 0.5882
Epoch 3/5
352/352 [==============================] - 5s 13ms/step - loss: 0.5617 - binary_crossentropy: 0.5617 - val_loss: 0.5521 - val_binary_crossentropy: 0.5521
Epoch 4/5
352/352 [==============================] - 5s 14ms/step - loss: 0.5399 - binary_crossentropy: 0.5399 - val_loss: 0.5458 - val_binary_crossentropy: 0.5458
Epoch 5/5
352/352 [==============================] - 5s 13ms/step - loss: 0.5348 - binary_crossentropy: 0.5348 - val_loss: 0.5434 - val_binary_crossentropy: 0.5434


In [14]:
# 6. 평가 (NDCG & HitRate)
print("\n========== Evaluation Start ==========")
user_pred_info = {}
top = 10

# 테스트 예측 수행
mymodel_user_pred_info = test_model(autoIntMLP_model, test_df, batch_size=batch_size)

# Top-10 추출
for user, data_info in tqdm(mymodel_user_pred_info.items(), desc="Ranking Predictions"):
    ranklist = sorted(data_info, key=lambda s : s[1], reverse=True)[:top]
    ranklist = list(dict.fromkeys([r[0] for r in ranklist]))
    user_pred_info[str(user)] = ranklist

# Ground Truth (실제 라벨 1인 영화들)
test_data = test_df[test_df['label']==1].groupby('user_id')['movie_id'].apply(list)

# 지표 계산
mymodel_ndcg_result = {}
mymodel_hitrate_result = {}

for user, data_info in tqdm(test_data.items(), desc="Calculating Metrics"):
    user_str = str(user)
    if user_str not in user_pred_info: continue
        
    mymodel_pred = user_pred_info[user_str]
    testset = list(set(np.array(data_info).astype(int)))
    mymodel_pred = mymodel_pred[:top]

    # NDCG & HitRate
    mymodel_ndcg_result[user] = get_NDCG(mymodel_pred, testset)
    mymodel_hitrate_result[user] = get_hit_rate(mymodel_pred, testset)

print("\n----------------------------------------------")
print(" AutoIntMLP model NDCG : ", round(np.mean(list(mymodel_ndcg_result.values())), 5))
print(" AutoIntMLP model HitRate : ", round(np.mean(list(mymodel_hitrate_result.values())), 5))
print("----------------------------------------------")


========== Evaluation Start ==========


Ranking Predictions: 100%|██████████| 6035/6035 [00:00<00:00, 75063.98it/s]
Calculating Metrics: 5994it [00:00, 6607.95it/s]


----------------------------------------------
 AutoIntMLP model NDCG :  0.66258
 AutoIntMLP model HitRate :  0.63291
----------------------------------------------


In [15]:
# 7. 모델 및 필수 파일 저장 (Streamlit용)
import os
# (1) Field Dims 저장
fd_path = os.path.expanduser('~/aiffel/autoint/field_dims.npy')
try:
    os.makedirs(os.path.dirname(fd_path), exist_ok=True)
    np.save(fd_path, field_dims)
    print(f"✅ field_dims 저장 완료 (덮어쓰기): {fd_path}")
except Exception as e:
    print(f"❌ field_dims 저장 실패: {e}")

# (2) 모델 가중치 저장 (.h5 형식)
weight_path = os.path.expanduser('~/aiffel/autoint/model/autoInt_model_weights.weights.h5')
try:
    os.makedirs(os.path.dirname(weight_path), exist_ok=True)
    autoIntMLP_model.save_weights(weight_path)
    print(f"✅ 모델 가중치 저장 완료 (덮어쓰기): {weight_path}")
except Exception as e:
    print(f"❌ 모델 가중치 저장 실패: {e}")

# (3) Label Encoder 저장
le_path = os.path.expanduser('~/aiffel/autoint/label_encoders.pkl')
try:
    joblib.dump(label_encoders, le_path)
    print(f"✅ LabelEncoder 저장 완료 (덮어쓰기): {le_path}")
except Exception as e:
    print(f"❌ LabelEncoder 저장 실패: {e}")

✅ field_dims 저장 완료 (덮어쓰기): /aiffel/aiffel/autoint/field_dims.npy
✅ 모델 가중치 저장 완료 (덮어쓰기): /aiffel/aiffel/autoint/model/autoInt_model_weights.weights.h5
✅ LabelEncoder 저장 완료 (덮어쓰기): /aiffel/aiffel/autoint/label_encoders.pkl


---

In [16]:
# ======================================================
# [긴급 수정] 이름표 초기화 및 재학습/저장 스크립트
# ======================================================
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from tensorflow.keras.layers import Layer, Dense, Flatten, Dropout, BatchNormalization, Activation
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import TruncatedNormal
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.model_selection import train_test_split

# 1. 뇌 청소 (이름표 초기화) - ★ 가장 중요! ★
tf.keras.backend.clear_session()
print("🧠 메모리 및 레이어 이름표 초기화 완료!")

# ------------------------------------------------------
# 2. 클래스 정의 (autoint_mlp.py와 완전히 동일하게 맞춤)
# ------------------------------------------------------
class FeaturesEmbedding(Layer):
    def __init__(self, field_dims, embed_dim, **kwargs):
        super(FeaturesEmbedding, self).__init__(**kwargs)
        self.total_dim = sum(field_dims)
        self.embed_dim = embed_dim
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.int32)
        self.embedding = tf.keras.layers.Embedding(input_dim=self.total_dim, output_dim=self.embed_dim)

    def build(self, input_shape):
        self.embedding.build(input_shape)
        self.embedding.set_weights([tf.keras.initializers.GlorotUniform()(shape=self.embedding.weights[0].shape)])

    def call(self, x):
        x = tf.cast(x, dtype=tf.int32)
        x = x + tf.constant(self.offsets)
        return self.embedding(x)

class MultiHeadSelfAttention(Layer):
    def __init__(self, att_embedding_size=8, head_num=2, use_res=True, scaling=False, seed=1024, **kwargs):
        super(MultiHeadSelfAttention, self).__init__(**kwargs)
        self.att_embedding_size = att_embedding_size
        self.head_num = head_num
        self.use_res = use_res
        self.seed = seed
        self.scaling = scaling

    def build(self, input_shape):
        embedding_size = int(input_shape[-1])
        self.W_Query = self.add_weight(name='query', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                       dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed))
        self.W_key = self.add_weight(name='key', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                     dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed + 1))
        self.W_Value = self.add_weight(name='value', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                       dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed + 2))
        if self.use_res:
            self.W_Res = self.add_weight(name='res', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                         dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed))
        super(MultiHeadSelfAttention, self).build(input_shape)

    def call(self, inputs, **kwargs):
        querys = tf.tensordot(inputs, self.W_Query, axes=(-1, 0))
        keys = tf.tensordot(inputs, self.W_key, axes=(-1, 0))
        values = tf.tensordot(inputs, self.W_Value, axes=(-1, 0))
        querys = tf.stack(tf.split(querys, self.head_num, axis=2))
        keys = tf.stack(tf.split(keys, self.head_num, axis=2))
        values = tf.stack(tf.split(values, self.head_num, axis=2))
        inner_product = tf.matmul(querys, keys, transpose_b=True)
        if self.scaling:
            inner_product /= self.att_embedding_size ** 0.5
        normalized_att_scores = tf.nn.softmax(inner_product)
        result = tf.matmul(normalized_att_scores, values)
        result = tf.concat(tf.split(result, self.head_num, ), axis=-1)
        result = tf.squeeze(result, axis=0)
        if self.use_res:
            result += tf.tensordot(inputs, self.W_Res, axes=(-1, 0))
        result = tf.nn.relu(result)
        return result

class AutoIntMLP(Layer):
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False, dnn_dropout=0.4, init_std=0.0001):
        super(AutoIntMLP, self).__init__()
        self.embedding = FeaturesEmbedding(field_dims, embedding_size)
        self.num_fields = len(field_dims)
        self.embedding_size = embedding_size
        self.final_layer = Dense(1, use_bias=False, kernel_initializer=tf.random_normal_initializer(stddev=init_std))
        
        self.dnn = tf.keras.Sequential()
        for units in dnn_hidden_units:
            self.dnn.add(Dense(units, activation=dnn_activation, kernel_regularizer=tf.keras.regularizers.l2(l2_reg_dnn), kernel_initializer=tf.random_normal_initializer(stddev=init_std)))
            if dnn_use_bn: self.dnn.add(BatchNormalization())
            self.dnn.add(Activation(dnn_activation))
            if dnn_dropout > 0: self.dnn.add(Dropout(dnn_dropout))
        self.dnn.add(Dense(1, kernel_initializer=tf.random_normal_initializer(stddev=init_std)))
        
        self.int_layers = [MultiHeadSelfAttention(att_embedding_size=embedding_size, head_num=att_head_num, use_res=att_res) for _ in range(att_layer_num)]

    def call(self, inputs):
        embed_x = self.embedding(inputs)
        dnn_embed = tf.reshape(embed_x, shape=(-1, self.embedding_size * self.num_fields))
        att_input = embed_x
        for layer in self.int_layers:
            att_input = layer(att_input)
        att_output = Flatten()(att_input)
        att_output = self.final_layer(att_output)
        dnn_output = self.dnn(dnn_embed)
        y_pred = tf.keras.activations.sigmoid(att_output + dnn_output)
        return y_pred

class AutoIntMLPModel(Model):
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2,
                 att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False,
                 dnn_dropout=0.4, init_std=0.0001):
        super(AutoIntMLPModel, self).__init__()
        self.autoInt_layer = AutoIntMLP(field_dims, embedding_size, att_layer_num, att_head_num, att_res, dnn_hidden_units, dnn_activation, l2_reg_dnn, l2_reg_embedding, dnn_use_bn, dnn_dropout, init_std)

    def call(self, inputs, training=False):
        return self.autoInt_layer(inputs, training=training)

# ------------------------------------------------------
# 3. 데이터 로드 및 훈련 실행
# ------------------------------------------------------
# 데이터 경로 (환경에 맞게 수정)
data_path = os.path.expanduser('~/aiffel/autoint/ml-1m')
movielens_rcmm = pd.read_csv(f"{data_path}/movielens_rcmm_v2.csv", dtype=str)

# 라벨 인코더 및 전처리
from sklearn.preprocessing import LabelEncoder
label_encoders = {col: LabelEncoder() for col in movielens_rcmm.columns[:-1]}
for col, le in label_encoders.items():
    movielens_rcmm[col] = le.fit_transform(movielens_rcmm[col])
movielens_rcmm['label'] = movielens_rcmm['label'].astype(np.float32)

# 학습 데이터 준비
train_df, _ = train_test_split(movielens_rcmm, test_size=0.2, random_state=42)
u_i_feature = ['user_id', 'movie_id']
meta_features = ['movie_decade', 'movie_year', 'rating_year', 'rating_month', 'rating_decade', 'genre1','genre2', 'genre3', 'gender', 'age', 'occupation', 'zip']
field_dims = np.max(movielens_rcmm[u_i_feature + meta_features].astype(np.int64).values, axis=0) + 1

# 모델 생성 및 학습
model = AutoIntMLPModel(field_dims=field_dims, embedding_size=16, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu', dnn_dropout=0.4)
model.compile(optimizer=Adam(learning_rate=0.0001), loss=BinaryCrossentropy(), metrics=['binary_crossentropy'])

print("🚀 학습 시작 (빠르게 완료됩니다)...")
model.fit(train_df[u_i_feature + meta_features], train_df['label'], epochs=5, batch_size=2048, validation_split=0.1)

# ------------------------------------------------------
# 4. 안전하게 저장 (이름표 꼬임 방지)
# ------------------------------------------------------
save_path = os.path.expanduser('~/aiffel/autoint/model/autoInt_model_weights.weights.h5')
os.makedirs(os.path.dirname(save_path), exist_ok=True)
model.save_weights(save_path)
print(f"✅ [저장 완료] 깨끗한 가중치 파일이 생성되었습니다: {save_path}")

🧠 메모리 및 레이어 이름표 초기화 완료!
🚀 학습 시작 (빠르게 완료됩니다)...
Epoch 1/5
352/352 [==============================] - 7s 15ms/step - loss: 0.6738 - binary_crossentropy: 0.6738 - val_loss: 0.6321 - val_binary_crossentropy: 0.6321
Epoch 2/5
352/352 [==============================] - 5s 13ms/step - loss: 0.6030 - binary_crossentropy: 0.6030 - val_loss: 0.5894 - val_binary_crossentropy: 0.5894
Epoch 3/5
352/352 [==============================] - 5s 13ms/step - loss: 0.5608 - binary_crossentropy: 0.5608 - val_loss: 0.5497 - val_binary_crossentropy: 0.5497
Epoch 4/5
352/352 [==============================] - 5s 13ms/step - loss: 0.5380 - binary_crossentropy: 0.5380 - val_loss: 0.5452 - val_binary_crossentropy: 0.5452
Epoch 5/5
352/352 [==============================] - 5s 13ms/step - loss: 0.5340 - binary_crossentropy: 0.5340 - val_loss: 0.5430 - val_binary_crossentropy: 0.5430
✅ [저장 완료] 깨끗한 가중치 파일이 생성되었습니다: /aiffel/aiffel/autoint/model/autoInt_model_weights.weights.h5


---

### 해결되지 않는 레이어 넘버 이슈..

In [1]:
# 라이브러리 임포트
import time
import random
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, MaxPooling2D, Conv2D, Dropout, Lambda, Dense, Flatten, Activation, Input, Embedding, BatchNormalization
from tensorflow.keras.initializers import glorot_normal, Zeros, TruncatedNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import BinaryAccuracy
from collections import defaultdict
import joblib
import math

In [2]:
# =================================================================================
# [이름표 고정 버전] autoint_mlp_train.ipynb 모델 정의 부분 교체
# =================================================================================
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Flatten, Dropout, BatchNormalization, Activation
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import TruncatedNormal
import numpy as np

# 1. 임베딩 레이어 (이름 고정)
class FeaturesEmbedding(Layer):
    def __init__(self, field_dims, embed_dim, **kwargs):
        # name 강제 지정
        if 'name' not in kwargs: kwargs['name'] = 'fixed_embedding_layer'
        super(FeaturesEmbedding, self).__init__(**kwargs)
        self.total_dim = sum(field_dims)
        self.embed_dim = embed_dim
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.int32)
        self.embedding = tf.keras.layers.Embedding(input_dim=self.total_dim, output_dim=self.embed_dim, name='emb_matrix')

    def build(self, input_shape):
        self.embedding.build(input_shape)
        self.embedding.set_weights([tf.keras.initializers.GlorotUniform()(shape=self.embedding.weights[0].shape)])

    def call(self, x):
        x = tf.cast(x, dtype=tf.int32)
        x = x + tf.constant(self.offsets)
        return self.embedding(x)

# 2. 어텐션 레이어 (이름 고정)
class MultiHeadSelfAttention(Layer):
    def __init__(self, att_embedding_size=8, head_num=2, use_res=True, scaling=False, seed=1024, **kwargs):
        super(MultiHeadSelfAttention, self).__init__(**kwargs)
        self.att_embedding_size = att_embedding_size
        self.head_num = head_num
        self.use_res = use_res
        self.seed = seed
        self.scaling = scaling

    def build(self, input_shape):
        embedding_size = int(input_shape[-1])
        # 가중치 이름도 명확하게 지정
        self.W_Query = self.add_weight(name='query_W', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                       dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed))
        self.W_key = self.add_weight(name='key_W', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                     dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed + 1))
        self.W_Value = self.add_weight(name='value_W', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                       dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed + 2))
        if self.use_res:
            self.W_Res = self.add_weight(name='res_W', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                         dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed))
        super(MultiHeadSelfAttention, self).build(input_shape)

    def call(self, inputs, **kwargs):
        querys = tf.tensordot(inputs, self.W_Query, axes=(-1, 0))
        keys = tf.tensordot(inputs, self.W_key, axes=(-1, 0))
        values = tf.tensordot(inputs, self.W_Value, axes=(-1, 0))
        querys = tf.stack(tf.split(querys, self.head_num, axis=2))
        keys = tf.stack(tf.split(keys, self.head_num, axis=2))
        values = tf.stack(tf.split(values, self.head_num, axis=2))
        inner_product = tf.matmul(querys, keys, transpose_b=True)
        if self.scaling:
            inner_product /= self.att_embedding_size ** 0.5
        normalized_att_scores = tf.nn.softmax(inner_product)
        result = tf.matmul(normalized_att_scores, values)
        result = tf.concat(tf.split(result, self.head_num, ), axis=-1)
        result = tf.squeeze(result, axis=0)
        if self.use_res:
            result += tf.tensordot(inputs, self.W_Res, axes=(-1, 0))
        result = tf.nn.relu(result)
        return result

# 3. 메인 레이어 (이름 고정)
class AutoIntMLP(Layer):
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False, dnn_dropout=0.4, init_std=0.0001, **kwargs):
        # name 강제 지정
        if 'name' not in kwargs: kwargs['name'] = 'fixed_autoint_mlp_layer'
        super(AutoIntMLP, self).__init__(**kwargs)
        self.embedding = FeaturesEmbedding(field_dims, embedding_size, name='fixed_embedding')
        self.num_fields = len(field_dims)
        self.embedding_size = embedding_size

        # Output Layer 이름 고정
        self.final_layer = Dense(1, use_bias=False, kernel_initializer=tf.random_normal_initializer(stddev=init_std), name='fixed_final_output')
        
        # DNN Layer 이름 고정
        self.dnn = tf.keras.Sequential(name='fixed_dnn_stack')
        for i, units in enumerate(dnn_hidden_units):
            self.dnn.add(Dense(units, activation=dnn_activation, 
                               kernel_regularizer=tf.keras.regularizers.l2(l2_reg_dnn),
                               kernel_initializer=tf.random_normal_initializer(stddev=init_std),
                               name=f'fixed_dnn_dense_{i}')) # <--- 여기가 핵심! 번호 고정
            if dnn_use_bn: self.dnn.add(BatchNormalization(name=f'fixed_dnn_bn_{i}'))
            self.dnn.add(Activation(dnn_activation, name=f'fixed_dnn_act_{i}'))
            if dnn_dropout > 0: self.dnn.add(Dropout(dnn_dropout, name=f'fixed_dnn_drop_{i}'))
        self.dnn.add(Dense(1, kernel_initializer=tf.random_normal_initializer(stddev=init_std), name='fixed_dnn_output'))
        
        # Attention Layer 이름 고정
        self.int_layers = [MultiHeadSelfAttention(att_embedding_size=embedding_size, head_num=att_head_num, use_res=att_res, name=f'fixed_attention_{i}') for i in range(att_layer_num)]

    def call(self, inputs):
        embed_x = self.embedding(inputs)
        dnn_embed = tf.reshape(embed_x, shape=(-1, self.embedding_size * self.num_fields))
        att_input = embed_x
        for layer in self.int_layers:
            att_input = layer(att_input)
        att_output = Flatten()(att_input)
        att_output = self.final_layer(att_output)
        dnn_output = self.dnn(dnn_embed)
        y_pred = tf.keras.activations.sigmoid(att_output + dnn_output)
        return y_pred

# 4. 모델 래퍼 (이름 고정)
class AutoIntMLPModel(Model):
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2,
                 att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False,
                 dnn_dropout=0.4, init_std=0.0001):
        super(AutoIntMLPModel, self).__init__(name='fixed_model_wrapper')
        self.autoInt_layer = AutoIntMLP(field_dims, embedding_size, att_layer_num, att_head_num, att_res, dnn_hidden_units, dnn_activation, l2_reg_dnn, l2_reg_embedding, dnn_use_bn, dnn_dropout, init_std)

    def call(self, inputs, training=False):
        return self.autoInt_layer(inputs, training=training)

In [3]:
# ------------------------------------------------------
# 데이터 로드 및 훈련 실행
# ------------------------------------------------------
# 데이터 경로 (환경에 맞게 수정)
data_path = os.path.expanduser('~/aiffel/autoint/ml-1m')
movielens_rcmm = pd.read_csv(f"{data_path}/movielens_rcmm_v2.csv", dtype=str)

# 라벨 인코더 및 전처리
from sklearn.preprocessing import LabelEncoder
label_encoders = {col: LabelEncoder() for col in movielens_rcmm.columns[:-1]}
for col, le in label_encoders.items():
    movielens_rcmm[col] = le.fit_transform(movielens_rcmm[col])
movielens_rcmm['label'] = movielens_rcmm['label'].astype(np.float32)

# 학습 데이터 준비
train_df, _ = train_test_split(movielens_rcmm, test_size=0.2, random_state=42)
u_i_feature = ['user_id', 'movie_id']
meta_features = ['movie_decade', 'movie_year', 'rating_year', 'rating_month', 'rating_decade', 'genre1','genre2', 'genre3', 'gender', 'age', 'occupation', 'zip']
field_dims = np.max(movielens_rcmm[u_i_feature + meta_features].astype(np.int64).values, axis=0) + 1

# 모델 생성 및 학습
model = AutoIntMLPModel(field_dims=field_dims, embedding_size=16, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu', dnn_dropout=0.4)
model.compile(optimizer=Adam(learning_rate=0.0001), loss=BinaryCrossentropy(), metrics=['binary_crossentropy'])

print("🚀 학습 시작 (빠르게 완료됩니다)...")
model.fit(train_df[u_i_feature + meta_features], train_df['label'], epochs=5, batch_size=2048, validation_split=0.1)

# ------------------------------------------------------
# 안전하게 저장
# ------------------------------------------------------
save_path = os.path.expanduser('~/aiffel/autoint/model/autoInt_model_weights.weights.h5')
os.makedirs(os.path.dirname(save_path), exist_ok=True)
model.save_weights(save_path)
print(f"✅ [저장 완료] 깨끗한 가중치 파일이 생성되었습니다: {save_path}")

🚀 학습 시작 (빠르게 완료됩니다)...
Epoch 1/5
352/352 [==============================] - 8s 15ms/step - loss: 0.6738 - binary_crossentropy: 0.6738 - val_loss: 0.6316 - val_binary_crossentropy: 0.6316
Epoch 2/5
352/352 [==============================] - 5s 13ms/step - loss: 0.6029 - binary_crossentropy: 0.6029 - val_loss: 0.5900 - val_binary_crossentropy: 0.5900
Epoch 3/5
352/352 [==============================] - 5s 13ms/step - loss: 0.5659 - binary_crossentropy: 0.5659 - val_loss: 0.5533 - val_binary_crossentropy: 0.5533
Epoch 4/5
352/352 [==============================] - 5s 13ms/step - loss: 0.5398 - binary_crossentropy: 0.5398 - val_loss: 0.5453 - val_binary_crossentropy: 0.5453
Epoch 5/5
352/352 [==============================] - 5s 14ms/step - loss: 0.5344 - binary_crossentropy: 0.5344 - val_loss: 0.5432 - val_binary_crossentropy: 0.5432
✅ [저장 완료] 깨끗한 가중치 파일이 생성되었습니다: /aiffel/aiffel/autoint/model/autoInt_model_weights.weights.h5


---

### 계속 해결되지 않는 모델 로드 ㅠㅠ

In [1]:
# [노트북용] 모델 정의부터 학습, 저장까지 한 번에 실행하는 코드
import os
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Flatten, Dropout, BatchNormalization, Activation, Embedding
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import TruncatedNormal, GlorotUniform
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. 초기화
tf.keras.backend.clear_session()

# 2. 클래스 정의 (로컬 autoint_mlp.py와 100% 동일)
class FeaturesEmbedding(Layer):
    def __init__(self, field_dims, embed_dim, **kwargs):
        if 'name' not in kwargs: kwargs['name'] = 'fixed_embedding_layer'
        super(FeaturesEmbedding, self).__init__(**kwargs)
        self.total_dim = sum(field_dims)
        self.embed_dim = embed_dim
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.int32)
        self.embedding = Embedding(input_dim=self.total_dim, output_dim=self.embed_dim, name='emb_matrix')
    def build(self, input_shape):
        self.embedding.build(input_shape)
        self.embedding.set_weights([GlorotUniform()(shape=self.embedding.weights[0].shape)])
    def call(self, x):
        x = tf.cast(x, dtype=tf.int32)
        x = x + tf.constant(self.offsets)
        return self.embedding(x)

class MultiHeadSelfAttention(Layer):
    def __init__(self, att_embedding_size=8, head_num=2, use_res=True, scaling=False, seed=1024, **kwargs):
        super(MultiHeadSelfAttention, self).__init__(**kwargs)
        self.att_embedding_size, self.head_num, self.use_res, self.seed, self.scaling = att_embedding_size, head_num, use_res, seed, scaling
    def build(self, input_shape):
        embedding_size = int(input_shape[-1])
        self.W_Query = self.add_weight(name='query_W', shape=[embedding_size, self.att_embedding_size * self.head_num], dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed))
        self.W_key = self.add_weight(name='key_W', shape=[embedding_size, self.att_embedding_size * self.head_num], dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed + 1))
        self.W_Value = self.add_weight(name='value_W', shape=[embedding_size, self.att_embedding_size * self.head_num], dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed + 2))
        if self.use_res: self.W_Res = self.add_weight(name='res_W', shape=[embedding_size, self.att_embedding_size * self.head_num], dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed))
        super(MultiHeadSelfAttention, self).build(input_shape)
    def call(self, inputs, **kwargs):
        querys = tf.tensordot(inputs, self.W_Query, axes=(-1, 0))
        keys = tf.tensordot(inputs, self.W_key, axes=(-1, 0))
        values = tf.tensordot(inputs, self.W_Value, axes=(-1, 0))
        querys = tf.stack(tf.split(querys, self.head_num, axis=2))
        keys = tf.stack(tf.split(keys, self.head_num, axis=2))
        values = tf.stack(tf.split(values, self.head_num, axis=2))
        inner_product = tf.matmul(querys, keys, transpose_b=True)
        if self.scaling: inner_product /= self.att_embedding_size ** 0.5
        normalized_att_scores = tf.nn.softmax(inner_product)
        result = tf.matmul(normalized_att_scores, values)
        result = tf.concat(tf.split(result, self.head_num, ), axis=-1)
        result = tf.squeeze(result, axis=0)
        if self.use_res: result += tf.tensordot(inputs, self.W_Res, axes=(-1, 0))
        return tf.nn.relu(result)

class AutoIntMLP(Layer):
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False, dnn_dropout=0.4, init_std=0.0001, **kwargs):
        if 'name' not in kwargs: kwargs['name'] = 'fixed_autoint_mlp_layer'
        super(AutoIntMLP, self).__init__(**kwargs)
        self.embedding = FeaturesEmbedding(field_dims, embedding_size, name='fixed_embedding')
        self.num_fields = len(field_dims)
        self.embedding_size = embedding_size
        self.final_layer = Dense(1, use_bias=False, kernel_initializer=tf.random_normal_initializer(stddev=init_std), name='fixed_final_output')
        
        # [수정] Sequential 제거
        self.dnn_layers = []
        for i, units in enumerate(dnn_hidden_units):
            self.dnn_layers.append(Dense(units, activation=None, kernel_regularizer=tf.keras.regularizers.l2(l2_reg_dnn), kernel_initializer=tf.random_normal_initializer(stddev=init_std), name=f'fixed_dnn_dense_{i}'))
            if dnn_use_bn: self.dnn_layers.append(BatchNormalization(name=f'fixed_dnn_bn_{i}'))
            self.dnn_layers.append(Activation(dnn_activation, name=f'fixed_dnn_act_{i}'))
            if dnn_dropout > 0: self.dnn_layers.append(Dropout(dnn_dropout, name=f'fixed_dnn_drop_{i}'))
        self.dnn_output_layer = Dense(1, kernel_initializer=tf.random_normal_initializer(stddev=init_std), name='fixed_dnn_output_layer')
        
        self.int_layers = [MultiHeadSelfAttention(att_embedding_size=embedding_size, head_num=att_head_num, use_res=att_res, name=f'fixed_attention_{i}') for i in range(att_layer_num)]

    def call(self, inputs):
        embed_x = self.embedding(inputs)
        dnn_embed = tf.reshape(embed_x, shape=(-1, self.embedding_size * self.num_fields))
        att_input = embed_x
        for layer in self.int_layers: att_input = layer(att_input)
        att_output = Flatten()(att_input)
        att_output = self.final_layer(att_output)
        
        dnn_output = dnn_embed
        for layer in self.dnn_layers: dnn_output = layer(dnn_output)
        dnn_output = self.dnn_output_layer(dnn_output)
        
        return tf.keras.activations.sigmoid(att_output + dnn_output)

class AutoIntMLPModel(Model):
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2,
                 att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False,
                 dnn_dropout=0.4, init_std=0.0001):
        super(AutoIntMLPModel, self).__init__(name='fixed_model_wrapper')
        self.autoInt_layer = AutoIntMLP(field_dims, embedding_size, att_layer_num, att_head_num, att_res, dnn_hidden_units, dnn_activation, l2_reg_dnn, l2_reg_embedding, dnn_use_bn, dnn_dropout, init_std)
    def call(self, inputs, training=False):
        return self.autoInt_layer(inputs, training=training)

# 3. 데이터 로드 및 학습 (약식)
data_path = os.path.expanduser('~/aiffel/autoint/ml-1m')
movielens_rcmm = pd.read_csv(f"{data_path}/movielens_rcmm_v2.csv", dtype=str)
from sklearn.preprocessing import LabelEncoder
label_encoders = {col: LabelEncoder() for col in movielens_rcmm.columns[:-1]}
for col, le in label_encoders.items(): movielens_rcmm[col] = le.fit_transform(movielens_rcmm[col])
movielens_rcmm['label'] = movielens_rcmm['label'].astype(np.float32)
train_df, _ = train_test_split(movielens_rcmm, test_size=0.2, random_state=42)
u_i_feature = ['user_id', 'movie_id']
meta_features = ['movie_decade', 'movie_year', 'rating_year', 'rating_month', 'rating_decade', 'genre1','genre2', 'genre3', 'gender', 'age', 'occupation', 'zip']
field_dims = np.max(movielens_rcmm[u_i_feature + meta_features].astype(np.int64).values, axis=0) + 1

model = AutoIntMLPModel(field_dims=field_dims, embedding_size=16, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu', dnn_dropout=0.4)
model.compile(optimizer=Adam(learning_rate=0.0001), loss=BinaryCrossentropy(), metrics=['binary_crossentropy'])
model.fit(train_df[u_i_feature + meta_features], train_df['label'], epochs=5, batch_size=2048, validation_split=0.1)

# 4. 저장
save_path = os.path.expanduser('~/aiffel/autoint/model/autoInt_model_weights.weights.h5')
os.makedirs(os.path.dirname(save_path), exist_ok=True)
model.save_weights(save_path)
print(f"✅ 저장 완료: {save_path}")

Epoch 1/5
352/352 [==============================] - 8s 15ms/step - loss: 0.6736 - binary_crossentropy: 0.6736 - val_loss: 0.6315 - val_binary_crossentropy: 0.6315
Epoch 2/5
352/352 [==============================] - 5s 13ms/step - loss: 0.6042 - binary_crossentropy: 0.6042 - val_loss: 0.5938 - val_binary_crossentropy: 0.5938
Epoch 3/5
352/352 [==============================] - 5s 14ms/step - loss: 0.5805 - binary_crossentropy: 0.5805 - val_loss: 0.5747 - val_binary_crossentropy: 0.5747
Epoch 4/5
352/352 [==============================] - 5s 13ms/step - loss: 0.5513 - binary_crossentropy: 0.5513 - val_loss: 0.5503 - val_binary_crossentropy: 0.5503
Epoch 5/5
352/352 [==============================] - 5s 14ms/step - loss: 0.5379 - binary_crossentropy: 0.5379 - val_loss: 0.5451 - val_binary_crossentropy: 0.5451
✅ 저장 완료: /aiffel/aiffel/autoint/model/autoInt_model_weights.weights.h5


---

ㅎㅏ ㅇ ㅏ ... ㅠㅠ

In [2]:
# =================================================================
# [긴급 복구] 로컬 코드와 100% 일치하는 가중치 파일 재생성 코드
# =================================================================
import os
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Flatten, Dropout, BatchNormalization, Activation, Embedding
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import TruncatedNormal, GlorotUniform
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. 뇌 청소 (이름표 초기화)
tf.keras.backend.clear_session()

# 2. 클래스 정의 (현재 로컬 autoint_mlp.py와 완벽 일치 버전)
class FeaturesEmbedding(Layer):
    def __init__(self, field_dims, embed_dim, **kwargs):
        if 'name' not in kwargs: kwargs['name'] = 'fixed_embedding_layer'
        super(FeaturesEmbedding, self).__init__(**kwargs)
        self.total_dim = sum(field_dims)
        self.embed_dim = embed_dim
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.int32)
        self.embedding = Embedding(input_dim=self.total_dim, output_dim=self.embed_dim, name='emb_matrix')
    def build(self, input_shape):
        self.embedding.build(input_shape)
        self.embedding.set_weights([GlorotUniform()(shape=self.embedding.weights[0].shape)])
    def call(self, x):
        x = tf.cast(x, dtype=tf.int32)
        x = x + tf.constant(self.offsets)
        return self.embedding(x)

class MultiHeadSelfAttention(Layer):
    def __init__(self, att_embedding_size=8, head_num=2, use_res=True, scaling=False, seed=1024, **kwargs):
        super(MultiHeadSelfAttention, self).__init__(**kwargs)
        self.att_embedding_size, self.head_num, self.use_res, self.seed, self.scaling = att_embedding_size, head_num, use_res, seed, scaling
    def build(self, input_shape):
        embedding_size = int(input_shape[-1])
        self.W_Query = self.add_weight(name='query_W', shape=[embedding_size, self.att_embedding_size * self.head_num], dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed))
        self.W_key = self.add_weight(name='key_W', shape=[embedding_size, self.att_embedding_size * self.head_num], dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed + 1))
        self.W_Value = self.add_weight(name='value_W', shape=[embedding_size, self.att_embedding_size * self.head_num], dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed + 2))
        if self.use_res: self.W_Res = self.add_weight(name='res_W', shape=[embedding_size, self.att_embedding_size * self.head_num], dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed))
        super(MultiHeadSelfAttention, self).build(input_shape)
    def call(self, inputs, **kwargs):
        querys = tf.tensordot(inputs, self.W_Query, axes=(-1, 0))
        keys = tf.tensordot(inputs, self.W_key, axes=(-1, 0))
        values = tf.tensordot(inputs, self.W_Value, axes=(-1, 0))
        querys = tf.stack(tf.split(querys, self.head_num, axis=2))
        keys = tf.stack(tf.split(keys, self.head_num, axis=2))
        values = tf.stack(tf.split(values, self.head_num, axis=2))
        inner_product = tf.matmul(querys, keys, transpose_b=True)
        if self.scaling: inner_product /= self.att_embedding_size ** 0.5
        normalized_att_scores = tf.nn.softmax(inner_product)
        result = tf.matmul(normalized_att_scores, values)
        result = tf.concat(tf.split(result, self.head_num, ), axis=-1)
        result = tf.squeeze(result, axis=0)
        if self.use_res: result += tf.tensordot(inputs, self.W_Res, axes=(-1, 0))
        return tf.nn.relu(result)

class AutoIntMLP(Layer):
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False, dnn_dropout=0.4, init_std=0.0001, **kwargs):
        if 'name' not in kwargs: kwargs['name'] = 'fixed_autoint_mlp_layer'
        super(AutoIntMLP, self).__init__(**kwargs)
        self.embedding = FeaturesEmbedding(field_dims, embedding_size, name='fixed_embedding')
        self.num_fields = len(field_dims)
        self.embedding_size = embedding_size
        self.final_layer = Dense(1, use_bias=False, kernel_initializer=tf.random_normal_initializer(stddev=init_std), name='fixed_final_output')
        
        # [핵심] Sequential 제거하고 리스트 사용 (현재 로컬 코드와 일치시킴)
        self.dnn_layers = []
        for i, units in enumerate(dnn_hidden_units):
            self.dnn_layers.append(Dense(units, activation=None, kernel_regularizer=tf.keras.regularizers.l2(l2_reg_dnn), kernel_initializer=tf.random_normal_initializer(stddev=init_std), name=f'fixed_dnn_dense_{i}'))
            if dnn_use_bn: self.dnn_layers.append(BatchNormalization(name=f'fixed_dnn_bn_{i}'))
            self.dnn_layers.append(Activation(dnn_activation, name=f'fixed_dnn_act_{i}'))
            if dnn_dropout > 0: self.dnn_layers.append(Dropout(dnn_dropout, name=f'fixed_dnn_drop_{i}'))
        self.dnn_output_layer = Dense(1, kernel_initializer=tf.random_normal_initializer(stddev=init_std), name='fixed_dnn_output_layer')
        
        self.int_layers = [MultiHeadSelfAttention(att_embedding_size=embedding_size, head_num=att_head_num, use_res=att_res, name=f'fixed_attention_{i}') for i in range(att_layer_num)]

    def call(self, inputs):
        embed_x = self.embedding(inputs)
        dnn_embed = tf.reshape(embed_x, shape=(-1, self.embedding_size * self.num_fields))
        att_input = embed_x
        for layer in self.int_layers: att_input = layer(att_input)
        att_output = Flatten()(att_input)
        att_output = self.final_layer(att_output)
        
        dnn_output = dnn_embed
        for layer in self.dnn_layers: dnn_output = layer(dnn_output)
        dnn_output = self.dnn_output_layer(dnn_output)
        return tf.keras.activations.sigmoid(att_output + dnn_output)

class AutoIntMLPModel(Model):
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu', l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False, dnn_dropout=0.4, init_std=0.0001):
        super(AutoIntMLPModel, self).__init__(name='fixed_model_wrapper')
        self.autoInt_layer = AutoIntMLP(field_dims, embedding_size, att_layer_num, att_head_num, att_res, dnn_hidden_units, dnn_activation, l2_reg_dnn, l2_reg_embedding, dnn_use_bn, dnn_dropout, init_std)
    def call(self, inputs, training=False):
        return self.autoInt_layer(inputs, training=training)

# 3. 데이터 로드 및 학습
data_path = os.path.expanduser('~/aiffel/autoint/ml-1m')
movielens_rcmm = pd.read_csv(f"{data_path}/movielens_rcmm_v2.csv", dtype=str)
label_encoders = {col: LabelEncoder() for col in movielens_rcmm.columns[:-1]}
for col, le in label_encoders.items(): movielens_rcmm[col] = le.fit_transform(movielens_rcmm[col])
movielens_rcmm['label'] = movielens_rcmm['label'].astype(np.float32)
train_df, _ = train_test_split(movielens_rcmm, test_size=0.2, random_state=42)
u_i_feature = ['user_id', 'movie_id']
meta_features = ['movie_decade', 'movie_year', 'rating_year', 'rating_month', 'rating_decade', 'genre1','genre2', 'genre3', 'gender', 'age', 'occupation', 'zip']
field_dims = np.max(movielens_rcmm[u_i_feature + meta_features].astype(np.int64).values, axis=0) + 1

model = AutoIntMLPModel(field_dims=field_dims, embedding_size=16, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu', dnn_dropout=0.4)
model.compile(optimizer=Adam(learning_rate=0.0001), loss=BinaryCrossentropy(), metrics=['binary_crossentropy'])

print("🚀 학습 시작 (5 Epoch)...")
model.fit(train_df[u_i_feature + meta_features], train_df['label'], epochs=5, batch_size=2048, validation_split=0.1)

# 4. 저장 (이름: autoInt_model_weights.weights.h5)
save_path = os.path.expanduser('~/aiffel/autoint/model/autoInt_model_weights.weights.h5')
os.makedirs(os.path.dirname(save_path), exist_ok=True)
model.save_weights(save_path)
print(f"✅ [최종 저장 완료] 이 파일을 다운로드하세요: {save_path}")

🚀 학습 시작 (5 Epoch)...
Epoch 1/5
352/352 [==============================] - 7s 15ms/step - loss: 0.6741 - binary_crossentropy: 0.6741 - val_loss: 0.6312 - val_binary_crossentropy: 0.6312
Epoch 2/5
352/352 [==============================] - 5s 13ms/step - loss: 0.6029 - binary_crossentropy: 0.6029 - val_loss: 0.5908 - val_binary_crossentropy: 0.5908
Epoch 3/5
352/352 [==============================] - 5s 13ms/step - loss: 0.5690 - binary_crossentropy: 0.5690 - val_loss: 0.5554 - val_binary_crossentropy: 0.5554
Epoch 4/5
352/352 [==============================] - 5s 13ms/step - loss: 0.5407 - binary_crossentropy: 0.5407 - val_loss: 0.5459 - val_binary_crossentropy: 0.5459
Epoch 5/5
352/352 [==============================] - 5s 13ms/step - loss: 0.5349 - binary_crossentropy: 0.5349 - val_loss: 0.5442 - val_binary_crossentropy: 0.5442
✅ [최종 저장 완료] 이 파일을 다운로드하세요: /aiffel/aiffel/autoint/model/autoInt_model_weights.weights.h5
